# Building & Measuring a RAG Pipeline (open-source, end to end)

This notebook builds a small **Retrieval-Augmented Generation** system with
fully open-source models, then measures it with the standard RAG metrics.

- **Embeddings / retriever:** `sentence-transformers` (`all-MiniLM-L6-v2`)
- **Generator (LLM):** `google/flan-t5-base` — small, CPU-friendly, instruction-tuned
- **Metrics we implement from scratch:** Hit Rate, MRR, Precision@k, Recall@k,
  NDCG@k (retrieval) and Faithfulness, Answer Relevancy, Context Relevance,
  Answer Correctness (generation).

> 📖 The **theory + flowcharts** for every metric live in the companion website:
> open `rag_eval/site/index.html` in a browser. This notebook is the hands-on
> counterpart — same metrics, real models, real numbers.

Everything runs on CPU. First run downloads ~300 MB of model weights.

## 0. Install dependencies

Run once. Restart the kernel afterwards if imports fail.

In [1]:
%pip install -q "sentence-transformers>=2.2" "transformers>=4.40" "torch>=2.0" numpy pandas

Note: you may need to restart the kernel to use updated packages.


## 0b. Silence the Hugging Face Xet progress-bar bug

The `hf_xet` download backend throws a harmless `LookupError` when it tries to
update its progress bar from a background thread inside Jupyter. The download
still works. Run this **before** any `transformers` / `sentence-transformers`
import to disable the buggy progress rendering.


In [2]:
import os
# Must be set BEFORE importing huggingface libraries.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # stops the xet LookupError spam
# Alternatively, use the classic (non-Xet) downloader:
# os.environ["HF_HUB_DISABLE_XET"] = "1"
print("HF progress bars disabled")


HF progress bars disabled


## 1. The knowledge base

A RAG system answers questions from a corpus. Here's a tiny one — each entry is
a "document" the retriever can fetch. In production these are chunks of your PDFs,
wiki pages, tickets, etc.

In [3]:
DOCUMENTS = {
    "d_paris":   "Paris is the capital and most populous city of France. Its population within the city limits is about 2.1 million people.",
    "d_france":  "France is a country in Western Europe. It is known for its cuisine, art, and the Eiffel Tower.",
    "d_hamlet":  "Hamlet is a tragedy written by William Shakespeare, believed to have been composed between 1599 and 1601.",
    "d_globe":   "The Globe Theatre in London was an open-air playhouse associated with William Shakespeare's acting company.",
    "d_python":  "Python is a high-level programming language created by Guido van Rossum and first released in 1991.",
    "d_everest": "Mount Everest is Earth's highest mountain above sea level, with a peak at 8,849 metres in the Himalayas.",
    "d_photo":   "Photosynthesis is the process by which green plants convert sunlight, water, and carbon dioxide into glucose and oxygen.",
    "d_water":   "Water is a chemical compound with the formula H2O: two hydrogen atoms bonded to one oxygen atom.",
}

DOC_IDS  = list(DOCUMENTS.keys())
DOC_TEXT = list(DOCUMENTS.values())
print(f"{len(DOCUMENTS)} documents loaded")

8 documents loaded


## 2. Embed the corpus & build a (tiny) vector index

We turn each document into a vector with a sentence-transformer, then retrieve
by **cosine similarity**. No vector database needed for 8 docs — NumPy is enough.
The concept is identical at scale; you'd just swap NumPy for FAISS/Chroma/pgvector.

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # ~90 MB

# normalize=True lets us use a plain dot product as cosine similarity.
doc_embeddings = embedder.encode(DOC_TEXT, normalize_embeddings=True)
print("doc embedding matrix:", doc_embeddings.shape)   # (8, 384)

2026-07-01 15:26:00.055598: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-01 15:26:00.097151: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-01 15:26:01.140378: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


doc embedding matrix: (8, 384)


## 3. The retriever

Embed the query, score it against every document, return the top-k by cosine
similarity. This is the "R" in RAG — and the thing the **retrieval metrics**
in section 6 grade.

In [5]:
def retrieve(query, k=3):
    """Return the top-k (doc_id, text, score), highest similarity first."""
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = doc_embeddings @ q                     # cosine sim, one per doc
    order = np.argsort(-scores)[:k]                 # top-k indices
    return [(DOC_IDS[i], DOC_TEXT[i], float(scores[i])) for i in order]

for did, text, score in retrieve("What is the capital of France?", k=3):
    print(f"{score:.3f}  {did}  ::  {text[:60]}...")

0.631  d_paris  ::  Paris is the capital and most populous city of France. Its p...
0.607  d_france  ::  France is a country in Western Europe. It is known for its c...
0.111  d_hamlet  ::  Hamlet is a tragedy written by William Shakespeare, believed...


## 4. The generator (open-source LLM)

`flan-t5-base` reads the retrieved context + question and writes an answer. This
is the "G" in RAG — graded by the **generation metrics** in section 7.

The prompt tells the model to answer **from the context only** — the grounding
instruction that Faithfulness will later check it actually obeyed.

In [6]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-base")  # ~300 MB

def generate(query, contexts, max_new_tokens=64):
    context_block = "\n".join(contexts)
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the context lacks the answer, say you don't know.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {query}\n"
        "Answer:"
    )
    out = generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"].strip()

print(generate("What is the capital of France?",
               ["Paris is the capital and most populous city of France."]))

Device set to use cpu


Paris


## 5. The full RAG pipeline

Retrieve → stuff context into the prompt → generate. One function.

In [7]:
def rag(query, k=3):
    hits = retrieve(query, k=k)
    contexts = [text for _id, text, _s in hits]
    answer = generate(query, contexts)
    return {
        "question": query,
        "answer": answer,
        "contexts": contexts,
        "retrieved_ids": [_id for _id, _t, _s in hits],
    }

result = rag("Who wrote Hamlet?")
print("Q:", result["question"])
print("A:", result["answer"])
print("retrieved:", result["retrieved_ids"])

Q: Who wrote Hamlet?
A: William Shakespeare
retrieved: ['d_hamlet', 'd_globe', 'd_python']


## 6. Retrieval metrics — did we fetch the right documents?

Deterministic, no model needed. They compare the retriever's ranked output
against **gold labels** (which doc IDs are truly relevant). Implementations
below match the formulas on the website exactly.

In [8]:
import math

def hit_rate(retrieved, relevant, k):
    relevant = set(relevant)
    return 1.0 if any(d in relevant for d in retrieved[:k]) else 0.0

def reciprocal_rank(retrieved, relevant):
    relevant = set(relevant)
    for i, d in enumerate(retrieved, start=1):      # 1-indexed rank
        if d in relevant:
            return 1.0 / i                          # first hit only -> MRR when averaged
    return 0.0

def precision_at_k(retrieved, relevant, k):
    relevant, topk = set(relevant), retrieved[:k]
    if not topk:
        return 0.0
    return sum(d in relevant for d in topk) / len(topk)

def recall_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    return len(set(retrieved[:k]) & relevant) / len(relevant)

def ndcg_at_k(retrieved, relevant, k):
    relevant, topk = set(relevant), retrieved[:k]
    dcg = sum((1.0 if d in relevant else 0.0) / math.log2(i + 1)
              for i, d in enumerate(topk, start=1))
    ideal = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal + 1))
    return dcg / idcg if idcg > 0 else 0.0

# quick check: relevant doc at rank 2 of 3
demo = ["d_x", "d_gold", "d_y"]
print("hit_rate   ", hit_rate(demo, {"d_gold"}, 3))       # 1.0
print("mrr        ", reciprocal_rank(demo, {"d_gold"}))   # 0.5
print("precision  ", round(precision_at_k(demo, {"d_gold"}, 3), 3))  # 0.333
print("recall     ", recall_at_k(demo, {"d_gold"}, 3))    # 1.0
print("ndcg       ", round(ndcg_at_k(demo, {"d_gold"}, 3), 3))       # 0.631

hit_rate    1.0
mrr         0.5
precision   0.333
recall      1.0
ndcg        0.631


## 7. Generation metrics — is the *answer* any good?

The website uses **Claude** as an LLM judge. Here we stay fully open-source and
build the same ideas from the models we already loaded:

| Metric | Open-source approach used here |
|---|---|
| **Faithfulness** | Split answer into sentences; ask `flan-t5` *yes/no* whether the context supports each. Score = supported / total. (LLM-as-judge, open-source judge.) |
| **Answer Relevancy** | Cosine similarity between the question and the answer embeddings. |
| **Context Relevance** | Fraction of retrieved chunks whose similarity to the question clears a threshold. |
| **Answer Correctness** | Cosine similarity between the answer and a ground-truth reference. |

These are lighter than the Claude versions but demonstrate the mechanics. Swap in
a bigger open model (or an NLI cross-encoder for faithfulness) for stronger scores.

In [9]:
import re

def _cos(a, b):
    va, vb = embedder.encode([a, b], normalize_embeddings=True)
    return float(np.dot(va, vb))

def _sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def faithfulness(contexts, answer):
    """LLM-as-judge with an OPEN-SOURCE judge: is each answer sentence
    supported by the retrieved context?  Score = supported / total."""
    context_block = "\n".join(contexts)
    sentences = _sentences(answer)
    if not sentences:
        return 1.0, []
    verdicts = []
    for s in sentences:
        prompt = (f"Context:\n{context_block}\n\n"
                  f"Statement: {s}\n\n"
                  "Is the statement fully supported by the context? Answer yes or no.")
        out = generator(prompt, max_new_tokens=4, do_sample=False)[0]["generated_text"]
        supported = out.strip().lower().startswith("yes")
        verdicts.append((supported, s))
    score = sum(v for v, _ in verdicts) / len(verdicts)
    return score, verdicts

def answer_relevancy(question, answer):
    return _cos(question, answer)                  # 0..1 (embeddings are non-negative-ish)

def context_relevance(question, contexts, threshold=0.3):
    if not contexts:
        return 0.0
    return sum(_cos(question, c) >= threshold for c in contexts) / len(contexts)

def answer_correctness(answer, ground_truth):
    return _cos(answer, ground_truth)

# demo on the hallucination example from the website
ctx = ["Hamlet is a tragedy written by William Shakespeare."]
ans = "Hamlet was written by William Shakespeare in 1601."
score, verdicts = faithfulness(ctx, ans)
print("faithfulness:", round(score, 2))
for ok, s in verdicts:
    print("  ", "OK " if ok else "NO ", s)

faithfulness: 0.0
   NO  Hamlet was written by William Shakespeare in 1601.


## 8. Evaluate the whole pipeline over a labeled set

We define a small evaluation set with **gold labels** (`relevant_ids`) and
**reference answers** (`ground_truth`), run RAG on each question, and compute
every metric. Then aggregate into a table.

In [10]:
EVAL_SET = [
    {"question": "What is the capital of France?",
     "relevant_ids": ["d_paris"],
     "ground_truth": "Paris is the capital of France."},
    {"question": "Who wrote Hamlet?",
     "relevant_ids": ["d_hamlet"],
     "ground_truth": "William Shakespeare wrote Hamlet."},
    {"question": "How tall is Mount Everest?",
     "relevant_ids": ["d_everest"],
     "ground_truth": "Mount Everest is 8,849 metres tall."},
    {"question": "Who created the Python programming language?",
     "relevant_ids": ["d_python"],
     "ground_truth": "Guido van Rossum created Python."},
    {"question": "What is the chemical formula of water?",
     "relevant_ids": ["d_water"],
     "ground_truth": "The chemical formula of water is H2O."},
]
print(len(EVAL_SET), "eval questions")

5 eval questions


In [11]:
import pandas as pd

K = 3
rows = []
for ex in EVAL_SET:
    out = rag(ex["question"], k=K)
    retr = out["retrieved_ids"]
    faith, _ = faithfulness(out["contexts"], out["answer"])
    rows.append({
        "question":        ex["question"][:32],
        "answer":          out["answer"][:40],
        # retrieval
        "hit@k":           hit_rate(retr, ex["relevant_ids"], K),
        "mrr":             round(reciprocal_rank(retr, ex["relevant_ids"]), 3),
        "prec@k":          round(precision_at_k(retr, ex["relevant_ids"], K), 3),
        "recall@k":        round(recall_at_k(retr, ex["relevant_ids"], K), 3),
        "ndcg@k":          round(ndcg_at_k(retr, ex["relevant_ids"], K), 3),
        # generation
        "faithful":        round(faith, 3),
        "ans_rel":         round(answer_relevancy(ex["question"], out["answer"]), 3),
        "ctx_rel":         round(context_relevance(ex["question"], out["contexts"]), 3),
        "correct":         round(answer_correctness(out["answer"], ex["ground_truth"]), 3),
    })

df = pd.DataFrame(rows)
df

,question,answer,hit@k,mrr,prec@k,recall@k,ndcg@k,faithful,ans_rel,ctx_rel,correct
0,What is the capital of France?,Paris,1.0,1.0,0.333,1.0,1.0,1.0,0.628,0.667,0.734
1,Who wrote Hamlet?,William Shakespeare,1.0,1.0,0.333,1.0,1.0,0.0,0.593,0.667,0.763
2,How tall is Mount Everest?,"8,849 metres",1.0,1.0,0.333,1.0,1.0,1.0,0.454,0.333,0.666
3,Who created the Python programmi,Guido van Rossum,1.0,1.0,0.333,1.0,1.0,1.0,0.143,0.333,0.581
4,What is the chemical formula of,H2O,1.0,1.0,0.333,1.0,1.0,1.0,0.357,0.333,0.636


### Dataset-level scores

The mean of each metric across the eval set — your single-number scorecard.

In [12]:
metric_cols = ["hit@k", "mrr", "prec@k", "recall@k", "ndcg@k",
               "faithful", "ans_rel", "ctx_rel", "correct"]
df[metric_cols].mean().round(3).to_frame("mean_score")

,mean_score
hit@k,1.000
mrr,1.000
prec@k,0.333
recall@k,1.000
ndcg@k,1.000
faithful,0.800
ans_rel,0.435
ctx_rel,0.467
correct,0.676


## 9. What the numbers tell you

- **Retrieval metrics high, generation low** → the retriever works; fix the
  generator (bigger model, better prompt, tighter grounding).
- **Faithfulness low** → the model is hallucinating beyond the context. This is
  the metric to watch in production — and it needs **no gold labels**, so you can
  run it on live traffic.
- **Context Relevance low** → your retriever is padding the prompt with junk;
  lower `k` or add a re-ranker.

### Where to go next
- Swap `flan-t5-base` for a stronger open model (e.g. `flan-t5-large`, or an
  instruct model via `transformers`) and watch Faithfulness / Correctness rise.
- Use an **NLI cross-encoder** (e.g. `cross-encoder/nli-deberta-v3-small`) for a
  more rigorous Faithfulness than yes/no prompting.
- For Answer Relevancy, implement the **reverse-question** method from the
  website: generate questions the answer addresses, embed, compare to the real
  question.

📖 Revisit `rag_eval/site/index.html` for the theory and flowcharts behind
every metric above.